# Case-wise and monthly hearing-rate comparison

This notebook recreates the supplied Stage 2 calculations from the current CSVs and compares them with the dashboard for the same district, court, case type, and period. It reports case-wise hearings per calendar month, case-wise hearings per working day, and monthly hearing throughput per working day.

The pasted Stage 2 code imports helper modules that were not included. This notebook makes its assumptions visible: DAYS_PER_MONTH = 30.4375; Stage 2 date recognition follows its year-prefix rule; elapsed working days use the dashboard's activity calendar; monthly denominators come from the configured working-day workbook. It does not use or modify a database.

Requires pandas, numpy, and openpyxl. Run from inside the project folder.

In [1]:
from pathlib import Path
import sys, re
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
while not (ROOT / 'config' / 'courts.json').exists():
    if ROOT == ROOT.parent:
        raise FileNotFoundError('Run this notebook from inside the project folder.')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from app.config import SOURCE_DATA_DIR, district_configurations
from app.bootstrap import source_root
from app.data_store import CsvStore
from app.domain.context import Context
from app.ingestion.raw import import_snapshot

# Change these to inspect another dashboard selection.
DISTRICT = 'Purba Medinipur'
COURT = 'CJM'
CASE_TYPE = 'Gr Case'
DAYS_PER_MONTH = 30.4375

district_config = next(d for d in district_configurations() if d['district'] == DISTRICT)
court_config = next(c for c in district_config['courts'] if c['id'] == COURT)
START_MONTH, END_MONTH = district_config['start'], district_config['end']
CUTOFF = (district_config.get('listing_cutoffs') or {}).get(COURT)
SOURCE_ROOT = source_root(SOURCE_DATA_DIR, district_config)
COURT_FOLDER = SOURCE_ROOT / court_config['folder']
print(f'{DISTRICT} · {COURT} · {CASE_TYPE} · filed {START_MONTH}–{END_MONTH} · cutoff {CUTOFF}')

Purba Medinipur · CJM · Gr Case · filed 2010-01–2026-06 · cutoff 2017-01-01


## Recreate Stage 2 inputs from CSVs

The supplied code counts distinct dates within each hearing CSV separately and adds the per-file counts. Blank dates fail the year-prefix condition and are ignored. The monthly aggregation maps hearing dates to case types without restricting cases to the selected filing cohort.

In [2]:
def read_csv(name):
    return pd.read_csv(COURT_FOLDER / name, dtype='string', keep_default_na=False)

type_lookup = read_csv('case_type_t.csv').set_index('case_type')
case_parts = []
for filename, archived in [('civil_t.csv', 0), ('civil_t_a.csv', 1)]:
    frame = read_csv(filename)[['cino', 'regcase_type', 'date_of_filing', 'date_of_decision']].copy()
    frame['is_archived'] = archived
    frame['filed'] = pd.to_datetime(frame['date_of_filing'], errors='coerce')
    frame['decided'] = pd.to_datetime(frame['date_of_decision'], errors='coerce')
    case_parts.append(frame)

cases = pd.concat(case_parts, ignore_index=True)
cases['case_type'] = cases['regcase_type'].map(type_lookup['type_name'])
cases['type_flag'] = cases['regcase_type'].map(type_lookup['type_flag'])
cases = cases.dropna(subset=['case_type', 'filed']).copy()

case_hearing_parts, month_hearing_parts = [], []
for filename in ('daily_proc.csv', 'daily_proc_a.csv'):
    rows = read_csv(filename)[['cino', 'todays_date']].copy()
    rows = rows.loc[rows['todays_date'].str.match(r'^[12]\d{3}-', na=False)].copy()
    case_hearing_parts.append(
        rows.groupby('cino', as_index=False)['todays_date'].nunique()
        .rename(columns={'todays_date': 'hearing_days'})
    )
    rows['month'] = rows['todays_date'].str.slice(0, 7)
    month_hearing_parts.append(
        rows.groupby(['cino', 'month'], as_index=False)['todays_date'].nunique()
        .rename(columns={'todays_date': 'hearing_days'})
    )

# As in Stage 2, distinct counts are computed per file and then summed.
hearing_by_case = (pd.concat(case_hearing_parts, ignore_index=True)
                   .groupby('cino', as_index=False)['hearing_days'].sum())
hearing_by_month = (pd.concat(month_hearing_parts, ignore_index=True)
                    .groupby(['cino', 'month'], as_index=False)['hearing_days'].sum())
cases = cases.merge(hearing_by_case, on='cino', how='left')
cases['hearing_days'] = cases['hearing_days'].fillna(0).astype(int)
case_type_by_cino = dict(zip(cases['cino'], cases['case_type']))
hearing_by_month['case_type'] = hearing_by_month['cino'].map(case_type_by_cino)
hearing_by_month = hearing_by_month.dropna(subset=['case_type']).copy()

print(f'Loaded {len(cases):,} case rows and {len(hearing_by_month):,} case-month hearing rows.')

Loaded 68,210 case rows and 451,444 case-month hearing rows.


## Dashboard benchmark and case-wise rates

The Stage 2 case-wise population is archived cases filed on or after the court cutoff, with a valid decision no earlier than filing. It does not apply the selected period end. The dashboard also limits the filing cohort and decision observation end, and only counts hearings from filing through decision.

In [3]:
store = CsvStore()
snapshot_result = import_snapshot(store, SOURCE_ROOT, district_config)
dashboard = Context(store, snapshot_result['snapshot'], COURT, START_MONTH, END_MONTH)
dashboard_type = next(r for r in dashboard.arrivals if r['case_type'] == CASE_TYPE)
TYPE_CODE = dashboard_type['code']
print(f'Dashboard filing-cohort cases for this type: {dashboard_type["filings"]:,}')

working_day_dates = np.asarray(dashboard.calendar, dtype='datetime64[D]')

def working_days_between(filed, decided):
    filed_day = np.datetime64(pd.Timestamp(filed).date(), 'D')
    decided_day = np.datetime64(pd.Timestamp(decided).date(), 'D')
    return (np.searchsorted(working_day_dates, decided_day, side='right')
            - np.searchsorted(working_day_dates, filed_day, side='right'))

def percentile_summary(values):
    return {p: float(np.percentile(values, p)) for p in (75, 90, 95)}

# This is the supplied casewise_rate_row calculation, using the current dashboard calendar.
def casewise_rate_row(case_type, side, completed):
    row = {'case_type': case_type, 'side': side}
    if completed.empty:
        return row, None
    elapsed_workdays = np.array(
        [working_days_between(case.filed, case.decided) for case in completed.itertuples()],
        dtype=float,
    )
    elapsed_days = (completed['decided'] - completed['filed']).dt.days.to_numpy(dtype=float)
    hearing_counts = completed['hearing_days'].to_numpy(dtype=float)

    use_workdays = elapsed_workdays >= 1
    rate_per_workday = hearing_counts[use_workdays] / elapsed_workdays[use_workdays]
    use_months = elapsed_days >= 1
    rate_per_month = hearing_counts[use_months] / (elapsed_days[use_months] / DAYS_PER_MONTH)

    if use_workdays.sum():
        p = percentile_summary(rate_per_workday)
        row.update(
            n_wd=int(use_workdays.sum()),
            wd_mean=round(float(rate_per_workday.mean()), 3),
            wd_median=round(float(np.median(rate_per_workday)), 3),
            wd_p75=round(p[75], 3), wd_p90=round(p[90], 3), wd_p95=round(p[95], 3),
        )
    if use_months.sum():
        p = percentile_summary(rate_per_month)
        row.update(
            n_mo=int(use_months.sum()),
            mo_mean=round(float(rate_per_month.mean()), 3),
            mo_median=round(float(np.median(rate_per_month)), 3),
            mo_p75=round(p[75], 3), mo_p90=round(p[90], 3), mo_p95=round(p[95], 3),
        )
    return row, rate_per_month.tolist() if len(rate_per_month) else None

stage2_completed = cases[
    cases['case_type'].eq(CASE_TYPE)
    & cases['is_archived'].eq(1)
    & cases['filed'].ge(pd.Timestamp(CUTOFF))
    & cases['decided'].notna()
    & cases['decided'].ge(cases['filed'])
].copy()
stage2_casewise, stage2_month_case_rates = casewise_rate_row(CASE_TYPE, 'Criminal', stage2_completed)
dashboard_mo = next(r for r in dashboard.measure('case_rate_mo') if r['code'] == TYPE_CODE)
dashboard_wd = next(r for r in dashboard.measure('case_rate_wd') if r['code'] == TYPE_CODE)

casewise_table = pd.DataFrame([
    {'Measure': 'Case-wise calendar-month rate', 'Method': 'Stage 2',
     'n': stage2_casewise.get('n_mo'), 'mean': stage2_casewise.get('mo_mean'),
     'median': stage2_casewise.get('mo_median'), 'p75': stage2_casewise.get('mo_p75'),
     'p90': stage2_casewise.get('mo_p90'), 'p95': stage2_casewise.get('mo_p95')},
    {'Measure': 'Case-wise calendar-month rate', 'Method': 'Dashboard',
     'n': dashboard_mo['n'], 'mean': round(dashboard_mo['mean'], 3),
     'median': round(dashboard_mo['median'], 3), 'p75': round(dashboard_mo['p75'], 3),
     'p90': round(dashboard_mo['p90'], 3), 'p95': round(dashboard_mo['p95'], 3)},
    {'Measure': 'Case-wise working-day rate', 'Method': 'Stage 2',
     'n': stage2_casewise.get('n_wd'), 'mean': stage2_casewise.get('wd_mean'),
     'median': stage2_casewise.get('wd_median'), 'p75': stage2_casewise.get('wd_p75'),
     'p90': stage2_casewise.get('wd_p90'), 'p95': stage2_casewise.get('wd_p95')},
    {'Measure': 'Case-wise working-day rate', 'Method': 'Dashboard',
     'n': dashboard_wd['n'], 'mean': round(dashboard_wd['mean'], 3),
     'median': round(dashboard_wd['median'], 3), 'p75': round(dashboard_wd['p75'], 3),
     'p90': round(dashboard_wd['p90'], 3), 'p95': round(dashboard_wd['p95'], 3)},
])
display(casewise_table)

stage2_rate_eligible = stage2_completed[
    (stage2_completed['decided'] - stage2_completed['filed']).dt.days >= 1
].copy()
dashboard_eligible_ids = {
    c['cino'] for c in dashboard.groups[TYPE_CODE]
    if c['filing'] >= CUTOFF and dashboard.resolved(c)
    and (pd.Timestamp(c['decision']) - pd.Timestamp(c['filing'])).days >= 1
}
stage2_only = stage2_rate_eligible[~stage2_rate_eligible['cino'].isin(dashboard_eligible_ids)]
print(f'Stage 2 rate-eligible cases: {len(stage2_rate_eligible):,}; dashboard: {dashboard_mo["n"]:,}')
print(f'Stage 2-only rate cases: {stage2_only["cino"].nunique():,}')
print('Among these, decisions after the dashboard end:',
      int((stage2_only['decided'] > pd.Timestamp(dashboard.last)).sum()),
      '| filings after the dashboard end:',
      int((stage2_only['filed'] > pd.Timestamp(dashboard.last)).sum()))
print(f'Unrounded monthly medians — Stage 2: {np.median(stage2_month_case_rates):.9f}; '
      f'dashboard: {dashboard_mo["median"]:.9f}')

Dashboard filing-cohort cases for this type: 31,136


,Measure,Method,n,mean,median,p75,p90,p95
0,Case-wise calendar-month rate,Stage 2,6816,0.481,0.142,0.246,0.607,1.268
1,Case-wise calendar-month rate,Dashboard,6736,0.484,0.142,0.248,0.609,1.268
2,Case-wise working-day rate,Stage 2,6816,0.019,0.006,0.010,0.025,0.053
3,Case-wise working-day rate,Dashboard,6736,0.019,0.006,0.010,0.025,0.053


Stage 2 rate-eligible cases: 6,816; dashboard: 6,736
Stage 2-only rate cases: 80
Among these, decisions after the dashboard end: 80 | filings after the dashboard end: 4
Unrounded monthly medians — Stage 2: 0.141618626; dashboard: 0.142231308


## Monthly hearing rate per working day

This recreates the Stage 2 monthly series and compares its median across months with the dashboard monthly-rate median. The dashboard counts hearings only for cases filed within the selected cohort; the supplied Stage 2 aggregation includes mapped cases from all filing years.

In [4]:
months = dashboard.months
month_index = {month: i for i, month in enumerate(months)}
stage2_rows = hearing_by_month[
    hearing_by_month['case_type'].eq(CASE_TYPE)
    & hearing_by_month['month'].isin(month_index)
]
stage2_counts = stage2_rows.groupby('month')['hearing_days'].sum().to_dict()
workdays = dashboard.monthly_denominators('hearing_rate')
stage2_month_rates = np.array([
    stage2_counts.get(m, 0) / workdays[m] for m in months
    if workdays[m] is not None and workdays[m] > 0
], dtype=float)

dashboard_monthly = next(r for r in dashboard.measure('monthly_rate') if r['code'] == TYPE_CODE)
dashboard_month_rates = np.array([v for v in dashboard_monthly['values'] if v is not None], dtype=float)

def monthly_summary(values):
    p = percentile_summary(values)
    return {'months': len(values), 'mean': round(float(np.mean(values)), 3),
            'median': round(float(np.median(values)), 3), 'p75': round(p[75], 3),
            'p90': round(p[90], 3), 'p95': round(p[95], 3)}

display(pd.DataFrame([
    {'Method': 'Stage 2', **monthly_summary(stage2_month_rates)},
    {'Method': 'Dashboard', **monthly_summary(dashboard_month_rates)},
]))

may = '2019-05'
may_workdays = workdays[may]
dashboard_may_rate = dashboard_monthly['values'][month_index[may]]
display(pd.DataFrame([
    {'Method': 'Stage 2 (all filing years)', 'hearing dates': stage2_counts.get(may, 0),
     'working days': may_workdays,
     'rate per working day': round(stage2_counts.get(may, 0) / may_workdays, 2)},
    {'Method': f'Dashboard ({START_MONTH}–{END_MONTH} filing cohort)',
     'hearing dates': round(dashboard_may_rate * may_workdays),
     'working days': may_workdays, 'rate per working day': round(dashboard_may_rate, 2)},
]))

,Method,months,mean,median,p75,p90,p95
0,Stage 2,198,48.848,52.146,82.513,93.362,96.484
1,Dashboard,198,40.843,41.743,71.293,81.163,87.313


,Method,hearing dates,working days,rate per working day
0,Stage 2 (all filing years),1818,25,72.72
1,Dashboard (2010-01–2026-06 filing cohort),1543,25,61.72


## How to interpret the result

Read the sample sizes with the medians. Equal values after rounding do not mean both methods use the same cases or hearing dates. Stage 2 has no selected-period end check for completed cases, and its monthly hearing-rate aggregation does not restrict filings to the selected cohort. Its case-wise hearing count also includes dated rows after decision; the dashboard excludes those from case-wise rates.

The notebook reads source CSVs and the configured working-day workbook. It writes no source, database, or report files.